# HW07 — Основы LLM / LLM fundamentals
100 баллов: код 80 + анализ 20. / 100 points: code 80 + analysis 20.
**Данные / Data:** [AG News](https://huggingface.co/datasets/SetFit/ag_news), реальные англоязычные новости, четыре категории: World, Sports, Business, Sci/Tech. Объект — текст новости; признак — текст, доступный при поступлении; цель — тема. / Real English news; an observation is an article, its text is available at arrival, and the target is its topic.

**Разбиение / Splits:** 4 демонстрации (по одной на класс) и 16 Validation из official train; 16 final Test из official test. Точные текстовые дубликаты исключаются между используемыми частями. Это маленький учебный поднабор, не официальный benchmark. Нет гарантии независимости событий/авторов и переноса во времени; пересечение с предобучением неизвестно. / Four demonstrations and sixteen validation articles from official train; sixteen final test articles from official test. Exclude exact text duplicates. This small teaching subset is not an official benchmark; event/author independence and temporal transfer are not established, and pretraining overlap is unknown.

**Модель / Model:** [google/flan-t5-small](https://huggingface.co/google/flan-t5-small), компактный instruction-tuned encoder-decoder, не decoder-only чат-LLM. Результаты не обобщать на все LLM. / A compact instruction-tuned encoder-decoder, not a decoder-only chat LLM. Do not generalize its scores to all LLMs.

**Colab CPU:** первоначальная загрузка модели и данных требует сети; API-ключ не нужен. Установка — первая ячейка; при предложении Colab перезапустить сеанс сделайте это и продолжите со второй ячейки. / Initial model/data downloads require internet; no API key. Install first; if Colab requests a runtime restart, restart and continue from the second cell.

Сдайте выполненный ноутбук с ФИО и группой в папку ДЗ; срок — матрица группы. / Submit an executed notebook with name and group to the homework folder; deadlines follow the group matrix.


In [ ]:
STUDENT_NAME = '' # Введите ФИО / Enter your full name.
STUDENT_GROUP = '' # Введите группу / Enter your group.


In [ ]:
import sys, subprocess # Доступ к текущему Python и установщику / Access the current Python and installer.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.46.3', 'datasets==3.1.0', 'sentencepiece==0.2.2', 'scikit-learn', 'pandas', 'matplotlib']) # Установить зависимости упражнения / Install exercise dependencies.


In [ ]:
import json, hashlib, platform, time # Журнал, хеши, версии и время / Logging, hashes, versions and timing.
import numpy as np # Массивы меток / Label arrays.
import pandas as pd # Таблицы результатов / Result tables.
import matplotlib.pyplot as plt # Матрица ошибок / Confusion-matrix plot.
import torch, transformers, sklearn # Модель и версии библиотек / Model and library versions.
from datasets import load_dataset # Загрузить реальные новости / Load real news.
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM # Классы encoder-decoder / Encoder-decoder classes.
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay # Метрики и визуализация / Metrics and visualization.
SEED = 2027 # Фиксированный выбор поднабора / Fixed subset selection.
MODEL_ID = 'google/flan-t5-small' # Открытая компактная модель / Compact open model.
MODEL_REV = '0fc9ddf78a1e988dac52e2dac162b0ede4fd74ab' # Зафиксировать версию весов / Pin the weights revision.
DATA_REV = 'ca5ba619eb034211db5f70932b6702efd21e7c73' # Зафиксировать версию датасета / Pin the dataset revision.
LABELS = ['World', 'Sports', 'Business', 'Sci/Tech'] # Порядок классов AG News / AG News class order.
torch.manual_seed(SEED) # Зафиксировать генератор PyTorch / Seed the PyTorch generator.
torch.set_num_threads(2) # Умеренное число потоков CPU / Limit CPU threads.
dataset = load_dataset('SetFit/ag_news', revision=DATA_REV) # Скачать официальный источник поднабора / Download the source dataset.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REV) # Токенизатор той же версии / Load the matching tokenizer.
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID, revision=MODEL_REV, use_safetensors=True).to('cpu').eval() # CPU и режим без dropout / Use CPU and disable dropout.


In [ ]:
used = set() # Уже использованные нормализованные тексты / Previously used normalized texts.
parts = {} # Хранить отдельные выборки / Store separate partitions.
for name, source, per_class in [('demonstrations', 'train', 1), ('validation', 'train', 4), ('test', 'test', 4)]: # Источники и квоты до эксперимента / Prespecified sources and quotas.
    indexed = dataset[source].add_column('original_id', list(range(len(dataset[source])))).shuffle(seed=SEED) # Сохранить исходные ID до перемешивания / Preserve original IDs before shuffling.
    chosen, counts = [], [0] * 4 # Список и счётчики классов / Selected rows and class counters.
    for row in indexed: # Просмотреть фиксированный порядок / Iterate in the fixed order.
        label = int(row['label']) # Целочисленная метка / Integer label.
        key = ' '.join(row['text'].lower().split()) # Нормализация для поиска точных дубликатов / Normalize for exact duplicate detection.
        if counts[label] >= per_class or key in used: # Не превышать квоту и не повторять текст / Respect quotas and avoid duplicates.
            continue # Перейти к следующей новости / Skip this article.
        chosen.append(dict(id=source + ':' + str(row['original_id']), text=row['text'], label=label)) # Зафиксировать происхождение наблюдения / Record observation provenance.
        used.add(key) # Не использовать этот текст повторно / Exclude this text from later partitions.
        counts[label] += 1 # Увеличить число объектов класса / Increment the class count.
        if counts == [per_class] * 4: # Все квоты заполнены / All quotas are filled.
            break # Завершить выбор части / Finish selecting this partition.
    assert counts == [per_class] * 4 # Проверить сбалансированность / Verify balance.
    parts[name] = chosen # Сохранить часть / Store the partition.
demo, validation, test_rows = [parts[k] for k in ['demonstrations', 'validation', 'test']] # Разделить роли данных / Separate data roles.
assert len({r['id'] for rows in parts.values() for r in rows}) == 36 # Проверить непересечение ID / Verify disjoint IDs.
print({k: len(v) for k, v in parts.items()}) # Показать размеры / Show partition sizes.


In [ ]:
def short_text(text, limit=48): # Ограничить текст токенами до сборки запроса / Bound article tokens before assembling a prompt.
    ids = tokenizer.encode(text, add_special_tokens=False) # Получить ID без служебных токенов / Encode without special tokens.
    return tokenizer.decode(ids[:limit], skip_special_tokens=True) # Вернуть начало статьи в заданном бюджете / Return the article prefix within budget.
def make_prompt(text, examples): # Собрать инструкцию и демонстрации / Assemble instructions and demonstrations.
    prompt = 'Classify the news topic. Choose exactly one label: World, Sports, Business, Sci/Tech.\n' # Задать задачу и формат / Specify task and format.
    for row in examples: # Только заранее выделенные демонстрации Train / Only prespecified training demonstrations.
        prompt += 'News: ' + short_text(row['text']) + '\nLabel: ' + LABELS[row['label']] + '\n' # Добавить пример с эталоном / Append a labeled demonstration.
    prompt += 'News: ' + short_text(text) + '\nLabel:' # Добавить текущую статью без эталона / Append the current article without its label.
    assert len(tokenizer.encode(prompt)) <= 384 # Не допустить скрытого усечения запроса / Prevent silent prompt truncation.
    return prompt # Передать готовый запрос / Return the complete prompt.
def generate_raw(text, examples): # Получить исходный текст ответа / Generate a raw response.
    inputs = tokenizer(make_prompt(text, examples), return_tensors='pt', truncation=False) # Токенизировать без скрытой обрезки / Tokenize without silent truncation.
    with torch.inference_mode(): # Не строить граф градиентов / Disable gradient tracking.
        output = model.generate(**inputs, max_new_tokens=8, do_sample=False, num_beams=1) # Greedy-вывод с фиксированным лимитом / Fixed-budget greedy decoding.
    return tokenizer.decode(output[0], skip_special_tokens=True) # Отделить ответ от служебных токенов / Decode the response without special tokens.


## Задание 1 / Exercise 1 — токены и бюджет
RU: Возьмите первую новость Validation. Выведите исходный текст, токены, число слов и токенов; покажите, сколько токенов останется после лимита 48. Сравните длину полного zero-shot и four-shot запросов. Объясните, почему длина увеличилась и что теряется при обрезке. Ожидаемый результат: числа из реального запуска и объяснение 3–5 предложений. Не заменяйте статью синтетической фразой.
EN: Inspect the first validation article. Print its text, token pieces, word/token counts and how many tokens survive the 48-token budget. Compare full zero-shot/four-shot prompt lengths. Explain the added context cost and what truncation removes. Deliver measured values and 3–5 explanatory sentences; do not substitute an invented article.

**Ваш ответ и код / Your answer and code:** добавьте ячейки ниже / add cells below.

## Задание 2 / Exercise 2 — формат и метрики
RU: Реализуйте parse_label: принимается только точное имя категории после strip и casefold, иначе −1. Реализуйте metric_report с accuracy, macro_f1 по классам 0–3 и valid_rate. INVALID нельзя выбрасывать. Проверьте на реальных Validation-ответах; приведите пример различия между ошибкой формата и ошибкой класса (если одного типа нет, сообщите это). Ожидается код и таблица трёх метрик.
EN: Implement parse_label using exact category matching after strip and casefold; return −1 otherwise. Implement accuracy, macro_f1 over classes 0–3 and valid_rate. Never drop INVALID. Check real validation responses and distinguish format errors from classification errors; explicitly report absent error types. Deliver code and the three-metric table.

In [ ]:
def parse_label(raw): # Разберите ответ по условию / Parse the response as specified.
    raise NotImplementedError('Exercise 2 / Задание 2') # Замените собственным решением / Replace with your solution.


In [ ]:
def metric_report(y_true, predicted): # Рассчитайте три метрики / Compute the three metrics.
    raise NotImplementedError('Exercise 2 / Задание 2') # Замените собственным решением / Replace with your solution.


In [ ]:
def select_variant(reports): # Выберите вариант по правилу / Apply the selection rule.
    raise NotImplementedError('Exercise 3 / Задание 3') # Замените собственным решением / Replace with your solution.


In [ ]:
def run_variant(rows, examples): # Прогнать фиксированный список новостей / Evaluate a fixed article list.
    records = [] # Сохранить сырые ответы для аудита / Retain raw responses for inspection.
    for row in rows: # Обработать каждый объект / Process each observation.
        raw = generate_raw(row['text'], examples) # Получить ответ без доступа к метке / Generate without providing the reference label.
        records.append(dict(id=row['id'], true=row['label'], raw=raw, pred=parse_label(raw))) # Сохранить эталон отдельно от входа / Store the reference separately from model input.
    return records # Вернуть все результаты без фильтрации ошибок / Return every result without filtering failures.


## Задание 3 / Exercise 3 — гипотеза и сравнение
RU: До candidate запишите прогноз действия четырёх примеров. Получите оба варианта на одинаковых 16 Validation ID. Сравните macro-F1 и valid_rate. Выберите four-shot только при строго лучшем macro-F1; при равенстве zero-shot. Сохраните selection до Test. Объясните результат, даже если кандидат хуже. Ожидается журнал выбора и 4–6 предложений.
EN: Before running the candidate, predict the effect of four demonstrations. Evaluate both prompts on the same 16 validation IDs. Compare macro-F1 and valid_rate. Select four-shot only on strict macro-F1 improvement; ties retain zero-shot. Save selection before Test. Explain the result even if the candidate loses. Deliver the selection record and 4–6 sentences.

In [ ]:
hypothesis = 'Четыре примера могут улучшить macro-F1 за счёт пояснения категорий, но улучшение не гарантировано. / Four demonstrations may clarify categories and improve macro-F1, but improvement is not guaranteed.' # Прогноз записан до сравнения / Prespecify the hypothesis.
validation_runs = {name: run_variant(validation, examples) for name, examples in [('zero_shot', []), ('four_shot', demo)]} # Одинаковые новости для обоих запросов / Use the same articles for both prompts.
reports = {name: metric_report([r['true'] for r in rows], [r['pred'] for r in rows]) for name, rows in validation_runs.items()} # Не удалять INVALID из метрик / Keep invalid responses in the metrics.
display(pd.DataFrame(reports).T) # Показать сопоставимые метрики / Display comparable metrics.
selected = select_variant(reports) # Зафиксировать правило выбора / Apply the frozen selection rule.
selection = dict(selected=selected, validation_reports=reports, demonstration_ids=[r['id'] for r in demo], model_revision=MODEL_REV, dataset_revision=DATA_REV, max_new_tokens=8, do_sample=False, article_token_budget=48, criterion='validation macro-F1; ties zero-shot') # Записать настройки до Test / Record configuration before Test.
print(json.dumps(selection, ensure_ascii=False, indent=2)) # Показать зафиксированное решение / Display the frozen decision.


## Задание 4 / Exercise 4 — ошибки и финальная проверка
RU: Покажите до трёх реальных ошибок выбранного варианта: ID, исходный текст, true, raw, parsed. Для каждой отдельно запишите наблюдение и гипотезу причины; при отсутствии ошибок явно сообщите об этом. Постройте confusion matrix со столбцом INVALID. Один раз оцените замороженный вариант на Test, сохраните llm_experiment.json. Назовите два ограничения обобщения и следующий эксперимент без подбора по текущему Test.
EN: Show up to three actual failures with ID, article, true label, raw output and parsed prediction. Separate observation from a possible cause for each; report explicitly if there are none. Plot a confusion matrix including INVALID. Evaluate the frozen variant once on Test and save llm_experiment.json. State two generalization limits and a follow-up that does not tune on the current Test.

**Контракт / Contract:** сохраните матрицу в `matrix`, размер 5×5, порядок `[0,1,2,3,-1]`; строки true, столбцы predicted. Последняя строка нулевая, поскольку истинного класса INVALID нет. / Store a 5×5 array in `matrix`, labels `[0,1,2,3,-1]`, true rows and predicted columns. The last row is zero because INVALID is not a true class.


**Ваш ответ и код / Your answer and code:** добавьте ячейки ниже / add cells below.

In [ ]:
test_records = run_variant(test_rows, demo if selected == 'four_shot' else []) # Единственный Test после выбора / One final evaluation after selection.
test_report = metric_report([r['true'] for r in test_records], [r['pred'] for r in test_records]) # Оценить все тестовые объекты / Score all test observations.
log = dict(selection=selection, ids={k: [r['id'] for r in v] for k, v in parts.items()}, validation_runs=validation_runs, test_records=test_records, test_report=test_report, versions=dict(python=platform.python_version(), transformers=transformers.__version__, torch=torch.__version__, sklearn=sklearn.__version__)) # Собрать воспроизводимый журнал / Assemble a reproducible run log.
with open('llm_experiment.json', 'w', encoding='utf-8') as handle: # Открыть локальный журнал / Open the local log file.
    json.dump(log, handle, ensure_ascii=False, indent=2) # Сохранить без выдуманной даты сдачи / Save without inventing a submission timestamp.
print(test_report) # Показать финальный результат без нового подбора / Show the final result without retuning.


In [ ]:
assert parse_label('  SPORTS ') == 1 # Допустима нормализация регистра и пробелов / Case and surrounding whitespace are allowed.
assert parse_label('The answer is Sports') == -1 # Объяснение вместо метки нарушает контракт / An explanation instead of a label is invalid.
assert parse_label('Sci/Tech') == 3 # Проверить название четвёртого класса / Check the fourth category name.
real = validation_runs['zero_shot'] # Проверять на реальных ответах / Use actual model outputs.
assert np.isclose(metric_report([r['true'] for r in real], [r['pred'] for r in real])['accuracy'], np.mean([r['true'] == r['pred'] for r in real])) # Независимая проверка accuracy / Independently verify accuracy.
assert select_variant({'zero_shot': reports['zero_shot'], 'four_shot': reports['zero_shot']}) == 'zero_shot' # Проверить ничью на реальном отчёте / Check ties using an actual report.
assert sum(matrix.sum(axis=1)) == len(validation) # Не потерять INVALID-объекты / Ensure invalid outputs were not dropped.
print('Checks passed / Проверки пройдены; это не оценка / this is not a grade') # Отделить самопроверку от оценки / Distinguish checks from grading.
